# Làm sạch dữ liệu địa điểm du lịch SoulViet

Notebook này tái tạo toàn bộ quá trình làm sạch `data-tourist-attraction-v2.csv` và xuất ra `new_data.csv`. File nguồn không bị chỉnh sửa.

Các xử lý chính:
- giữ `Id` làm định danh nội bộ duy nhất cho từng địa điểm;
- loại các cột quản trị, UUID phân loại, cột hoàn toàn rỗng, cột hằng và nội dung AI bị lặp;
- giải mã tọa độ WKB thành `Lat` và `Lng`;
- tách thông tin ảnh từ `MediaInfo`;
- chuẩn hóa Unicode, khoảng trắng và JSON;
- chuyển mã `VibeTag` thành nhãn có ý nghĩa;
- coi điểm đánh giá bằng 0 khi chưa có review là dữ liệu thiếu;
- kiểm tra trùng lặp, miền giá trị và tính hợp lệ của JSON.

In [ ]:
from pathlib import Path
import json
import re
import struct
import sys
import unicodedata

import pandas as pd

# Hoạt động cả khi mở Jupyter từ thư mục dự án hoặc từ new_data_soulviet.
cwd = Path.cwd()
data_dir = cwd if (cwd / 'data-tourist-attraction-v2.csv').exists() else cwd / 'new_data_soulviet'
source_path = data_dir / 'data-tourist-attraction-v2.csv'
output_path = data_dir / 'new_data.csv'
project_root = data_dir.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.opening_hours import parse_operation_hours
from utils.visit_duration import estimate_visit_duration
assert source_path.exists(), f'Không tìm thấy file nguồn: {source_path.resolve()}'
source_path.resolve(), output_path.resolve()

## 1. Đọc và khảo sát dữ liệu gốc

In [ ]:
df = pd.read_csv(source_path, na_values=['NULL'], keep_default_na=True)
print(f'Kích thước: {df.shape[0]:,} dòng x {df.shape[1]} cột')
display(pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing': df.isna().sum(),
    'missing_%': (df.isna().mean() * 100).round(2),
    'unique': df.nunique(dropna=True),
}))
display(df.head(3))

## 2. Hàm làm sạch

`Location` là WKB dạng hex (Point có SRID). Hai số thực 64-bit cuối lần lượt là kinh độ và vĩ độ. Nhãn trải nghiệm được lấy từ `AiContext`; bảng ánh xạ chỉ là phương án dự phòng.

In [ ]:
def clean_text(value):
    if not isinstance(value, str):
        return value
    return unicodedata.normalize('NFC', value).strip()

def decode_wkb_point(hex_value):
    try:
        raw = bytes.fromhex(hex_value)
        lng, lat = struct.unpack('<dd', raw[-16:])
        return lat, lng
    except (TypeError, ValueError, struct.error):
        return None, None

def parse_json_dict(value):
    try:
        parsed = json.loads(value)
        return parsed if isinstance(parsed, dict) else {}
    except (TypeError, json.JSONDecodeError):
        return {}

def normalize_json_list(value):
    try:
        parsed = json.loads(value)
        if not isinstance(parsed, list):
            parsed = []
    except (TypeError, json.JSONDecodeError):
        parsed = []
    return json.dumps(parsed, ensure_ascii=False, separators=(',', ':'))

VIBE_FALLBACK = {
    1: 'Chữa lành & Yên bình',
    2: 'Năng động & Phiêu lưu',
    4: 'Sáng tạo & Truyền cảm hứng',
    5: 'Ẩm thực & Đặc sản',
    6: 'Đậm văn hóa & Bản địa',
}

def extract_vibe(ai_context, numeric_vibe):
    match = re.search(r'Tag trải nghiệm:\s*(.*?)\.\s*Ngân sách:', ai_context or '')
    return clean_text(match.group(1)) if match else VIBE_FALLBACK.get(numeric_vibe)

## 3. Biến đổi dữ liệu

In [ ]:
cleaned = df.copy()

text_columns = [
    'Id', 'PlaceId', 'Name', 'Type', 'Address', 'Description',
    'OperationHours',
]
for column in text_columns:
    cleaned[column] = cleaned[column].map(clean_text)

coordinates = cleaned['Location'].map(decode_wkb_point)
cleaned['Lat'] = coordinates.map(lambda pair: pair[0])
cleaned['Lng'] = coordinates.map(lambda pair: pair[1])

media = cleaned['MediaInfo'].map(parse_json_dict)
cleaned['MainImage'] = media.map(lambda item: clean_text(item.get('MainImage')) or None)
cleaned['LandImages_JSON'] = media.map(
    lambda item: json.dumps(item.get('LandImages') or [], ensure_ascii=False, separators=(',', ':'))
)

cleaned['VibeTag'] = [
    extract_vibe(context, vibe)
    for context, vibe in zip(cleaned['AiContext'], cleaned['VibeTag'])
]

for column in ['AllTypes', 'Activities', 'TopReviews']:
    cleaned[column] = cleaned[column].map(normalize_json_list)

# 0/5 không phải một đánh giá thực khi địa điểm chưa có lượt review.
no_reviews = cleaned['ReviewCount'].eq(0)
cleaned.loc[no_reviews & cleaned['RatingScore'].eq(0), 'RatingScore'] = pd.NA

# Giữ nguyên OperationHours và thêm phiên bản có cấu trúc cho graph/planner.
opening_hours = cleaned['OperationHours'].map(parse_operation_hours)
cleaned['OpeningHours_JSON'] = opening_hours.map(
    lambda item: json.dumps(item, ensure_ascii=False, separators=(',', ':'))
)
cleaned['OpeningHoursStatus'] = opening_hours.map(lambda item: item['status'])
cleaned['OpeningHoursNeedsReview'] = opening_hours.map(lambda item: item['needs_review'])
cleaned['OpeningHoursVerificationStatus'] = opening_hours.map(
    lambda item: 'unknown' if item['status'] == 'unknown' else (
        'needs_review' if item['needs_review'] else 'source_unverified'
    )
)

visit_duration = [
    estimate_visit_duration(primary_type, all_types)
    for primary_type, all_types in zip(cleaned['Type'], cleaned['AllTypes'])
]
cleaned['VisitDurationMinutes'] = [item['minutes'] for item in visit_duration]
cleaned['VisitDurationSource'] = [item['source'] for item in visit_duration]
cleaned['VisitDurationConfidence'] = [item['confidence'] for item in visit_duration]

output_columns = [
    'Id', 'PlaceId', 'Name', 'Type', 'AllTypes', 'Address', 'Lat', 'Lng',
    'RatingScore', 'ReviewCount', 'OperationHours', 'OpeningHours_JSON',
    'OpeningHoursStatus', 'OpeningHoursNeedsReview',
    'OpeningHoursVerificationStatus',
    'VisitDurationMinutes', 'VisitDurationSource', 'VisitDurationConfidence',
    'Description',
    'Activities', 'TopReviews', 'VibeTag', 'MainImage',
    'LandImages_JSON',
]
result = cleaned[output_columns].copy()
print(f'Kết quả: {result.shape[0]:,} dòng x {result.shape[1]} cột')
display(result.head(3))

## 4. Kiểm tra chất lượng

Các dòng thiếu `PlaceId`, đánh giá hoặc ảnh chính không bị xóa tự động vì những trường còn lại vẫn hữu ích.

In [ ]:
json_columns = ['AllTypes', 'Activities', 'TopReviews', 'LandImages_JSON']
json_errors = {}
for column in json_columns:
    errors = 0
    for value in result[column]:
        try:
            if not isinstance(json.loads(value), list):
                errors += 1
        except (TypeError, json.JSONDecodeError):
            errors += 1
    json_errors[column] = errors

opening_hours_errors = 0
for value in result['OpeningHours_JSON']:
    try:
        parsed = json.loads(value)
        if not isinstance(parsed, dict) or 'days' not in parsed:
            opening_hours_errors += 1
    except (TypeError, json.JSONDecodeError):
        opening_hours_errors += 1

quality_report = pd.Series({
    'Số dòng': len(result),
    'Dòng trùng hoàn toàn': int(result.duplicated().sum()),
    'Trùng Name + Address': int(result.duplicated(['Name', 'Address']).sum()),
    'Thiếu Id': int(result['Id'].isna().sum()),
    'Id bị trùng': int(result['Id'].duplicated().sum()),
    'Thiếu PlaceId': int(result['PlaceId'].isna().sum()),
    'Thiếu RatingScore': int(result['RatingScore'].isna().sum()),
    'Thiếu MainImage': int(result['MainImage'].isna().sum()),
    'JSON lỗi': sum(json_errors.values()),
    'Lat nhỏ nhất': result['Lat'].min(),
    'Lat lớn nhất': result['Lat'].max(),
    'Lng nhỏ nhất': result['Lng'].min(),
    'Lng lớn nhất': result['Lng'].max(),
})
quality_report.loc['OperationHours unknown'] = int(result['OpeningHoursStatus'].eq('unknown').sum())
quality_report.loc['OperationHours cần kiểm tra'] = int(result['OpeningHoursNeedsReview'].sum())
quality_report.loc['OperationHours lỗi parser'] = int(opening_hours.map(lambda item: bool(item['parse_errors'])).sum())
display(quality_report.to_frame('Giá trị'))
display(result['VibeTag'].value_counts().to_frame('Số lượng'))
assert result.duplicated().sum() == 0
assert result['Id'].notna().all() and result['Id'].is_unique
assert sum(json_errors.values()) + opening_hours_errors == 0
assert len(opening_hours) == 972
assert result['OpeningHoursStatus'].eq('unknown').sum() == 187
assert opening_hours.map(lambda item: bool(item['parse_errors'])).sum() == 0
assert result['Lat'].between(-90, 90).all()
assert result['Lng'].between(-180, 180).all()

## 5. Xuất file

Dùng UTF-8 có BOM (`utf-8-sig`) để tiếng Việt hiển thị đúng cả trong Excel.

In [ ]:
result.to_csv(output_path, index=False, encoding='utf-8-sig', na_rep='')
print(f'Đã tạo: {output_path.resolve()}')
print(f'Dung lượng: {output_path.stat().st_size / 1024 / 1024:.2f} MB')